# TT/MPS基礎 06 — Mixed-Canonical Form

## 今回の位置づけ

前回までに、

$$
\text{TT-SVD}
\rightarrow
\text{gauge freedom}
\rightarrow
\text{左直交化}
\rightarrow
\text{右直交化}
\rightarrow
\text{左右部分収縮の直交性}
$$

まで確認しました。

今回は、中心を第2サイトに置いた3階 TT

$$
X
=
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}
$$

を扱います。

### 今回やること

1. 左右の QR で残った $R_1,\;R_3^T$ を第2コアへ吸収する
2. 中心コア $G_2^{[C]}$ の shape と再構成不変性を確認する
3. mixed-canonical の左右の直交条件を確認する
4. $\|X\|_F=\|G_2^{[C]}\|_F$ を数式と PyTorch で確認する
5. 中心コアの変化 $\Delta G_2^{[C]}$ と全テンソルの変化 $\Delta X$ の関係を確認する

**orthogonality center の詳しい意味、中心移動、SVD、Schmidt 分解、TT rounding、MPO、DMRG にはまだ進みません。**

今回も、理論と検証条件までは示しますが、実装部分は `TODO` として残します。

## 1. 前回までの結果を用意する

3階 TT

$$
X=G_1G_2G_3
$$

の各コアを

$$
G_1\in\mathbb{R}^{1\times n_1\times r_1},
\qquad
G_2\in\mathbb{R}^{r_1\times n_2\times r_2},
\qquad
G_3\in\mathbb{R}^{r_2\times n_3\times 1}
$$

とします。

前回までに学んだ QR 直交化を使うと、

$$
G_1=G_1^{[L]}R_1
$$

$$
G_3=R_3^TG_3^{[R]}
$$

と書けます。

今回は mixed-canonical form に集中するため、この左右の QR 自体はセットアップで済ませます。

shape は

$$
G_1^{[L]}:(1,n_1,r_1),
\qquad
R_1:(r_1,r_1),
$$

$$
R_3^T:(r_2,r_2),
\qquad
G_3^{[R]}:(r_2,n_3,1)
$$

です。

In [ ]:
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)


def reconstruct_tt3(G1: torch.Tensor, G2: torch.Tensor, G3: torch.Tensor) -> torch.Tensor:
    """3個のTTコアから3階テンソルを再構成する。"""
    return torch.einsum("aib,bjc,ckd->ijk", G1, G2, G3)


# 小さい3階TT
n1, n2, n3 = 4, 3, 5
r1, r2 = 2, 3

G1 = torch.randn(1, n1, r1)
G2 = torch.randn(r1, n2, r2)
G3 = torch.randn(r2, n3, 1)

X_before = reconstruct_tt3(G1, G2, G3)

# --- 前回までに学習済みの左右QRをセットアップとして実行 ---

# 第1コア: left QR
A1 = G1.squeeze(0)                    # (n1, r1)
Q1, R1 = torch.linalg.qr(A1, mode="reduced")
G1_left = Q1.unsqueeze(0)             # (1, n1, r1)

# 第3コア: right QR
A3 = G3.squeeze(-1)                   # (r2, n3)
Q3, R3 = torch.linalg.qr(A3.T, mode="reduced")
G3_right = Q3.T.unsqueeze(-1)         # (r2, n3, 1)

print("G1_left:", tuple(G1_left.shape))
print("R1     :", tuple(R1.shape))
print("G2     :", tuple(G2.shape))
print("R3.T   :", tuple(R3.T.shape))
print("G3_right:", tuple(G3_right.shape))


G1_left: (1, 4, 2)
R1     : (2, 2)
G2     : (2, 3, 3)
R3.T   : (3, 3)
G3_right: (3, 5, 1)


## 2. なぜ中心コアへ集められるのか

元の TT に

$$
G_1=G_1^{[L]}R_1,
\qquad
G_3=R_3^TG_3^{[R]}
$$

を代入すると、

$$
X
=
G_1^{[L]}
R_1
G_2
R_3^T
G_3^{[R]}
$$

となります。

中央の

$$
R_1G_2R_3^T
$$

は、第2コアの左ボンドと右ボンドにだけ作用します。

そこで、この3つをまとめたテンソルを中心コア

$$
G_2^{[C]}
$$

として扱います。

成分で考えると、

- $R_1$ は第1–第2コア間のボンドを縮約する
- $R_3^T$ は第2–第3コア間のボンドを縮約する
- 物理添字 $i_2$ はそのまま残る

という構造になります。

## 3. 演習1 — 中心コア $G_2^{[C]}$ を作る

### TODO

1. `R1` を `G2` の左ボンドへ作用させる
2. `R3.T` をその結果の右ボンドへ作用させる
3. `G2_center` を作る
4. shape が
   $$
   (r_1,n_2,r_2)
   $$
   のままであることを確認する
5. `G1_left`, `G2_center`, `G3_right` から全テンソルを再構成し、元の `X_before` と比較する

### 考えること

数式

$$
G_2^{[C]}
=
R_1G_2R_3^T
$$

で、$R_1$ と $R_3^T$ はそれぞれ `G2` の**どの軸**と縮約するでしょうか。

また、左右の正方行列を吸収しても $G_2^{[C]}$ の外部 shape が変わらない理由も確認してください。

In [14]:
# TODO 1:
# 中心コア G2_center = R1 G2 R3.T を作ってください。
#
# 1. R1 と G2 を左ボンドで縮約
G2_center_left =torch.tensordot(R1,G2,dims=([1], [0]))
#
# 2. その結果と R3.T を右ボンドで縮約
G2_center = torch.tensordot(G2_center_left,R3.T,dims=([2], [0]))
#
# 3. shape を確認
print("G2_center shape:",tuple(G2_center.shape))
#
# 4. mixed-canonical 形から全テンソルを再構成
X_mixed = reconstruct_tt3(G1_left, G2_center , G3_right)
#
# 5. 元の X_before との差を確認
reconstruction_error = torch.norm(X_before-X_mixed)
#
print("mixed-canonical reconstruction error =", reconstruction_error)

print("TODO: G2_center を作り、全テンソルが不変か確認する")


G2_center shape: (2, 3, 3)
mixed-canonical reconstruction error = tensor(8.4268e-15)
TODO: G2_center を作り、全テンソルが不変か確認する


## 4. Mixed-canonical の左右の直交条件

中心を第2サイトに置いた形は

$$
X
=
G_1^{[L]}
G_2^{[C]}
G_3^{[R]}
$$

です。

ここで、

- $G_1^{[L]}$ は左直交
- $G_3^{[R]}$ は右直交
- $G_2^{[C]}$ には、通常は左・右どちらの直交条件も課さない

という配置になります。

左側を

$$
L_1(i_1,\alpha_1)
=
G_1^{[L]}(1,i_1,\alpha_1)
$$

右側を

$$
R_2(\alpha_2,i_3)
=
G_3^{[R]}(\alpha_2,i_3,1)
$$

と書くと、左右の条件は

$$
L_1^TL_1=I_{r_1},
\qquad
R_2R_2^T=I_{r_2}
$$

です。

この左右の正規直交基底の間に、一般形の中心コア $G_2^{[C]}$ が置かれます。

## 5. 演習2 — 左右の Gram 行列を確認する

### TODO

1. `G1_left` から行列 `L1` を作る
2. `G3_right` から行列 `R2` を作る
3. 左側について
   $$
   L_1^TL_1
   $$
   を計算する
4. 右側について
   $$
   R_2R_2^T
   $$
   を計算する
5. それぞれ対応する単位行列との差を Frobenius ノルムで確認する

### 考えること

左側ではボンド添字 $\alpha_1$ が**列**にあり、右側ではボンド添字 $\alpha_2$ が**行**にあります。

そのため、Gram 行列で転置が付く位置が左右で逆になります。

In [11]:
# TODO 2:
# mixed-canonical form の左右の直交性を数値確認してください。
#
L1 = G1_left.squeeze(0)  
R2 = G3_right.squeeze(-1)
#
left_gram = L1.T@L1
right_gram = R2@R2.T
#
I_r1 = torch.eye(left_gram.shape[1], dtype=left_gram.dtype, device=left_gram.device)
I_r2 = torch.eye(right_gram.shape[1], dtype=right_gram.dtype, device=right_gram.device)
#
left_error = torch.norm(I_r1-left_gram)
right_error = torch.norm(I_r2-right_gram)
#
print("L1.T @ L1 =",left_gram)
print("R2 @ R2.T =",right_gram)
print("left orthogonality error =", left_error)
print("right orthogonality error =", right_error)

print("TODO: mixed-canonical の左右の直交性を確認する")


L1.T @ L1 = tensor([[1.0000e+00, 6.9389e-17],
        [6.9389e-17, 1.0000e+00]])
R2 @ R2.T = tensor([[ 1.0000e+00,  3.4694e-17, -7.6328e-17],
        [ 3.4694e-17,  1.0000e+00, -2.7756e-17],
        [-7.6328e-17, -2.7756e-17,  1.0000e+00]])
left orthogonality error = tensor(2.4276e-16)
right orthogonality error = tensor(4.0437e-16)
TODO: mixed-canonical の左右の直交性を確認する


## 6. なぜ全テンソルのノルムが中心コアへ局所化するのか

mixed-canonical form を添字で書くと、

$$
X(i_1,i_2,i_3)
=
\sum_{\alpha_1,\alpha_2}
L_1(i_1,\alpha_1)
G_2^{[C]}(\alpha_1,i_2,\alpha_2)
R_2(\alpha_2,i_3)
$$

です。

Frobenius ノルムの二乗は

$$
\|X\|_F^2
=
\sum_{i_1,i_2,i_3}
X(i_1,i_2,i_3)^2
$$

です。

この式に mixed-canonical 表現を2回代入すると、

- 左側では $L_1^TL_1$
- 右側では $R_2R_2^T$

が現れます。

左右が直交しているため、これらはそれぞれ単位行列になり、異なるボンド添字どうしのクロス項が消えます。

次の演習では、まず数式でこの流れを追い、そのあと PyTorch でもノルムが一致することを確認します。

## 7. 演習3 — $\|X\|_F=\|G_2^{[C]}\|_F$ を確認する

### TODO

まず数式で、次の順に導出してください。

1. $\|X\|_F^2$ に mixed-canonical 表現を2回代入する
2. 2つ目のボンド添字を
   $$
   \alpha_1',\alpha_2'
   $$
   と区別する
3. $i_1$ に関する和へ左直交性を使う
4. $i_3$ に関する和へ右直交性を使う
5. Kronecker のデルタにより、どの添字が一致するか確認する
6. 最後に残る和が中心コアの Frobenius ノルムになることを確認する

その後、PyTorch で

$$
\|X\|_F
$$

と

$$
\|G_2^{[C]}\|_F
$$

を直接比較してください。

### 注意

最終式だけでなく、**左右の直交性によってどのクロス項が消えるか**を意識してください。

In [15]:
# TODO 3:
# 上の導出を整理したあと、ノルムの一致を数値で確認してください。
#
# mixed-canonical では左右が直交なので ||X||_F = ||G2_center||_F
X_norm = torch.linalg.vector_norm(X_mixed).item()
G2_center_norm = torch.linalg.vector_norm(G2_center).item()
norm_difference = abs(X_norm - G2_center_norm)

print("||X||_F =", X_norm)
print("||G2_center||_F =", G2_center_norm)
print("difference =", norm_difference)


||X||_F = 26.721800341545464
||G2_center||_F = 26.721800341545457
difference = 7.105427357601002e-15


## 8. 中心コアだけを変える

ここでは左右の正準コア

$$
G_1^{[L]},
\qquad
G_3^{[R]}
$$

を固定します。

中心コアだけを

$$
G_2^{[C]}
\longrightarrow
G_2^{[C]}+\Delta G_2^{[C]}
$$

と変えることを考えます。

変化後のテンソルを $\widetilde X$ とすると、

$$
\Delta X
=
\widetilde X-X
$$

です。

TT の縮約は各コアについて線形なので、中心コアの変化だけを取り出すと、

$$
\Delta X
$$

も左右の同じ直交基底にはさまれた形になります。

したがって、前節と同じ直交性が使えます。

## 9. 演習4 — $\Delta G_2^{[C]}$ と $\Delta X$ の等長性を確認する

### TODO

1. `G2_center` と同じ shape の小さい摂動 `delta_G2_center` を作る
2. 中心コアだけを変えた新しいテンソル `X_perturbed` を再構成する
3. 
   $$
   \Delta X
   =
   X_{\mathrm{perturbed}}-X_{\mathrm{mixed}}
   $$
   を作る
4. 
   $$
   \|\Delta X\|_F
   $$
   と
   $$
   \|\Delta G_2^{[C]}\|_F
   $$
   を比較する
5. 両者の差が丸め誤差水準になるか確認する

### 考えること

この結果は、固定した左右の正準コアによる写像

$$
\mathcal{F}:
\Delta G_2^{[C]}
\mapsto
\Delta X
$$

が Frobenius ノルムを保存することに対応します。

ここでは「等長」という言葉を、**入力の変化量と全テンソル側の変化量の長さが一致する**という意味で確認してください。

In [12]:
# TODO 4:
# 中心コアの摂動と全テンソルの摂動のノルムを比較してください。
#
# 1. 小さい摂動を作る
delta_G2_center = 1e-3 * torch.randn_like(G2_center)
#
# 2. 中心コアだけを変える
G2_center_perturbed = G2_center + delta_G2_center
#
# 3. 全テンソルを再構成（左右コアは固定）
X_perturbed = reconstruct_tt3(G1_left, G2_center_perturbed, G3_right)
#
# 4. 全テンソル側の変化
delta_X = X_perturbed - X_mixed
#
# 5. ノルムを比較
delta_X_norm = torch.linalg.vector_norm(delta_X).item()
delta_G2_center_norm = torch.linalg.vector_norm(delta_G2_center).item()
isometry_error = abs(delta_X_norm - delta_G2_center_norm)

print("||delta_X||_F =", delta_X_norm)
print("||delta_G2_center||_F =", delta_G2_center_norm)
print("isometry error =", isometry_error)


||delta_X||_F = 0.0024798186584537576
||delta_G2_center||_F = 0.0024798186584533556
isometry error = 4.0202216555762504e-16


## 10. 今回の到達点

今回の流れは、

$$
G_1G_2G_3
\rightarrow
G_1^{[L]}
\left(R_1G_2R_3^T\right)
G_3^{[R]}
$$

です。

自分の実装と導出で、

- $G_2^{[C]}$ を作っても全テンソルが不変
- 左側は左直交、右側は右直交
- $\|X\|_F=\|G_2^{[C]}\|_F$
- $\|\Delta X\|_F=\|\Delta G_2^{[C]}\|_F$

を確認できれば、このNotebookは完了です。

ここまでで **mixed-canonical form の数学的な基本性質**を一区切りにします。

次のテーマは orthogonality center ですが、このNotebookではまだ進みません。